# Gimlet Heterogeneous Precision/Width — Non-Record Proof Runbook

**Goal:** Train for 600 steps on a T4 GPU, save `final_model.pt` + `final_model.int8.ptz`, and compute BPB.

The training script (`train_gpt.py`) has **built-in T4 detection** — no manual regex patching needed.

| Cell | Purpose | Est. Time |
|------|---------|-----------|
| 1 | Setup: clone, deps, data | 3-5 min |
| 2 | Train 600 steps and emit final summary files | 30-50 min |
| 3 | Verify exact non-record criteria and write submission-ready summary | 2 min |

In [ ]:
# ====== CELL 1: SETUP (CLEAN RESET + DATA) ======
import os, shutil

%cd /content
for d in ['pg', 'openai-pg']:
    if os.path.exists(d):
        shutil.rmtree(d)

print("Cloning fresh repositories...")
!git clone -b feat/gimlet-hetero-v1 https://github.com/jmoncayo-pursuit/parameter-golf-gimlet-hetero.git pg
!git clone https://github.com/openai/parameter-golf.git openai-pg

%pip install -q torch numpy sentencepiece zstandard huggingface_hub

print("\n--- Downloading Tokenizer and 1 Data Shard ---")
%cd /content/openai-pg
!python3 data/cached_challenge_fineweb.py --train-shards 1

# Quick Verification
print("\n--- Verifying Setup ---")
tok_path = '/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model'
if os.path.exists(tok_path):
    print(f"✅ Tokenizer found: {tok_path}")
else:
    print(f"❌ Tokenizer MISSING at {tok_path}")

if os.path.exists('/content/pg/train_gpt.py'):
    print("✅ Fresh script downloaded.")
else:
    print("❌ Download failed.")

In [ ]:
# ====== MINIMAL PATCH: PHASE MARKERS + VAL_MAX_TOKENS ======
from pathlib import Path
import re

p = Path('/content/pg/train_gpt.py')
text = p.read_text()

# 1. Wire VAL_MAX_TOKENS
text = text.replace(
    'val_loss_every = int(os.environ.get("VAL_LOSS_EVERY", 1000))',
    'val_loss_every = int(os.environ.get("VAL_LOSS_EVERY", 1000))\n    val_max_tokens = int(os.environ.get("VAL_MAX_TOKENS", 0))'
)

text = text.replace(
    'val_tokens = load_validation_tokens(args.val_files, args.train_seq_len)',
    'val_tokens = load_validation_tokens(args.val_files, args.train_seq_len)\n        if args.val_max_tokens > 0:\n            val_tokens = val_tokens[:min(val_tokens.numel(), args.val_max_tokens + 1)]'
)

# 2. Inject Phase Markers into existing logic
# Training complete
text = text.replace(
    'if master_process:\n        log0(f"step:{step}/{args.iterations}',
    'log0(f"phase:training_complete steps:{step}/{args.iterations}")\n    if master_process:\n        log0(f"step:{step}/{args.iterations}'
)

# Serialization start
text = text.replace(
    'if master_process:\n        torch.save(base_model.state_dict(), "final_model.pt")',
    'if master_process:\n        log0("phase:serialization_start writing final_model.pt")\n        torch.save(base_model.state_dict(), "final_model.pt")'
)

# Quantization start
text = text.replace(
    'quant_obj, quant_stats = quantize_state_dict_int8',
    'log0("phase:quantization_start building int8+zlib artifact")\n    quant_obj, quant_stats = quantize_state_dict_int8'
)

# Roundtrip eval start
text = text.replace(
    'with open("final_model.int8.ptz", "rb") as f:',
    'log0("phase:roundtrip_eval_start loading quantized artifact and running final validation")\n    with open("final_model.int8.ptz", "rb") as f:'
)

# Done
text = text.replace(
    'log0(f"final_int8_zlib_roundtrip_exact',
    'log0("phase:done")\n    log0(f"final_int8_zlib_roundtrip_exact'
)

p.write_text(text)
print("✅ Minimal patches applied. Verifying syntax...")
!python3 -m py_compile /content/pg/train_gpt.py && echo "--- SYNTAX OK ---"
!grep -n "phase:" /content/pg/train_gpt.py

In [ ]:
# ====== CELL 2: VERIFIED PROOF RUN ======
import os, subprocess, time

%cd /content/pg

# Configure environment strictly as requested
env = os.environ.copy()
env.update({
    'NO_COMPILE': '1',
    'ITERATIONS': '300',
    'TRAIN_SEQ_LEN': '128',
    'TRAIN_BATCH_TOKENS': '8192',
    'VAL_LOSS_EVERY': '0',
    'TRAIN_LOG_EVERY': '10',
    'WARMUP_STEPS': '2',
    'MAX_WALLCLOCK_SECONDS': '0',
    'VAL_MAX_TOKENS': '65536',
    'DATA_PATH': '/content/openai-pg/data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH': '/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model',
    'PYTHONUNBUFFERED': '1'
})

print(f'--- Starting Proof Run ---')
print(f'Start Time: {time.strftime("%H:%M:%S")}\n')

proc = subprocess.Popen(
    ['python3', '-u', 'train_gpt.py'],
    cwd='/content/pg', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n[INTERRUPTED] Terminating training process...')

rc = proc.wait()
print(f'\n--- Training finished with exit code {rc} at {time.strftime("%H:%M:%S")} ---\n')

# Artifact verification
print('--- Artifact Status ---')
for name in ['final_model.pt', 'final_model.int8.ptz', 'final_summary.json', 'final_summary.md']:
    path = f'/content/pg/{name}'
    if os.path.exists(path):
        print(f'✅ {name}: {os.path.getsize(path)} bytes')
    else:
        print(f'❌ {name} NOT FOUND')

In [ ]:
# ====== CELL 2B: DEFINITIVE 600-STEP PROOF RUN ======
import os, subprocess, time

%cd /content/pg

# Configure environment for the 600-step definitive run
env = os.environ.copy()
env.update({
    'NO_COMPILE': '1',
    'ITERATIONS': '600',       # Target 600 steps
    'TRAIN_SEQ_LEN': '128',
    'TRAIN_BATCH_TOKENS': '8192',
    'VAL_LOSS_EVERY': '0',
    'TRAIN_LOG_EVERY': '10',
    'WARMUP_STEPS': '2',
    'MAX_WALLCLOCK_SECONDS': '0',
    'VAL_MAX_TOKENS': '65536', # Capped for speed on T4
    'DATA_PATH': '/content/openai-pg/data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH': '/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model',
    'PYTHONUNBUFFERED': '1'
})

print(f'--- Starting 600-Step Definitive Proof Run ---')
print(f'Start Time: {time.strftime("%H:%M:%S")}\n')

proc = subprocess.Popen(
    ['python3', '-u', 'train_gpt.py'],
    cwd='/content/pg', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n[INTERRUPTED] Terminating training process...')

rc = proc.wait()
print(f'\n--- Training finished with exit code {rc} at {time.strftime("%H:%M:%S")} ---')

# Artifact verification
print('\n--- Artifact Verification ---')
for name in ['final_model.pt', 'final_model.int8.ptz', 'final_summary.json', 'final_summary.md']:
    path = f'/content/pg/{name}'
    if os.path.exists(path):
        print(f'✅ {name}: {os.path.getsize(path)} bytes')
    else:
        print(f'❌ {name} NOT FOUND')


In [ ]:
# ====== CELL 3: VERIFY EXACT NON-RECORD CRITERIA ======
import json, pathlib

root = pathlib.Path('/content/pg')
summary_path = root / 'final_summary.json'
int8_path = root / 'final_model.int8.ptz'
ckpt_path = root / 'final_model.pt'

print('--- 1. FILE CHECK ---')
for path in [ckpt_path, int8_path, summary_path]:
    if path.exists():
        print(f'✅ {path.name}: {path.stat().st_size} bytes')
    else:
        print(f'❌ {path.name} MISSING')

if not int8_path.exists():
    raise FileNotFoundError('final_model.int8.ptz not found. Run Cell 2B first.')

code_size = (root / 'train_gpt.py').stat().st_size
int8_size = int8_path.stat().st_size
total = code_size + int8_size
limit_ok = total <= 16_000_000

print('\n--- 2. ARTIFACT SIZE ---')
print(f'train_gpt.py: {code_size} bytes')
print(f'final_model.int8.ptz: {int8_size} bytes')
print(f'Total submission size: {total} bytes')
print(f'16 MB limit: {'PASS' if limit_ok else 'FAIL'}')

print('\n--- 3. SUMMARY ---')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(f"Steps completed: {summary.get('iterations_completed', 'N/A')}/{summary.get('iterations_target', 'N/A')}")
    print(f"Roundtrip val_loss: {summary.get('roundtrip_val_loss', 'N/A')}")
    print(f"Roundtrip val_bpb: {summary.get('roundtrip_val_bpb', 'N/A')}")
else:
    print('final_summary.json missing; use the training log values directly.')

status = 'READY FOR NON-RECORD SUBMISSION' if limit_ok else 'OVER SIZE LIMIT'
print(f'\nStatus: {status}')
